# Test de la fonction d'association de station à des échangeurs

In [1]:
from datetime import date
from pathlib import Path

from enum import StrEnum
import sys
import folium
import json
import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from constants import TILES_IGN, ATTR_IGN

CHUNK_SIZE: int = 20 #0
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75
RAYON_ECHANGEUR = 3000
BUFFER_ECHANGEUR = 100

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,7, 25)
#day = date(2026,8, 1)
#day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

class NaturePool(StrEnum):
    AIRE = "Aire"
    ECHANGEUR = "Echangeur"
    ALL = "All"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../saturation/data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../saturation/source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics


In [3]:
def stations_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the nearest echangeur for each station within a maximum distance."""
    stations = stations.drop_duplicates(subset=[ID_STATION])
    ech_geo = gpd.GeoDataFrame(echangeurs, geometry=gpd.points_from_xy(echangeurs["longitude"], echangeurs["latitude"]), crs="EPSG:4326")
    stations_geo = gpd.GeoDataFrame(
        stations, 
        geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]), 
        crs="EPSG:4326")
   
    ech_station = gpd.sjoin_nearest(
        stations_geo[[ID_STATION, "geometry"]].to_crs(epsg=2154),
        ech_geo[[ID_POOL, "geometry"]].to_crs(epsg=2154),
        how="inner",
        max_distance=RAYON_ECHANGEUR,
        distance_col="distance",
    )
    return ech_station[[ID_STATION, "geometry", ID_POOL, "distance"]].reset_index(drop=True).to_crs(epsg=4326)

In [ ]:
stations = pd.DataFrame({
    ID_STATION: ["st1", "st2", "st3"],
    "latitude": [46.5, 46.6, 46.7],
    "longitude": [2.5, 2.6, 2.7]})
echangeurs = 

result = [['FRHPCPNF059704TOTEMSA', 'E000001'],
  ['FRTSLP436970', 'E000001'],
  ['FRELCP12953864', 'E000001'],
  ['FRHPCPNF059705TOTEM', 'E000001'],
  ['FRGYMP735569', 'E000001'],
  ['FRELCP12953963', 'E000001'],
  ['FRVIAP122064', 'E000001'],
  ['FRGYMP735571', 'E000003'],
  ['FRFASP11568718', 'E000003'],
  ['FRIOYP13530862', 'E000003'],
  ['FRPD1PKFCWTZ', 'E000003'],
  ['FRFASP11568719', 'E000003'],
  ['FRECHPCRXMB', 'E000003'],
  ['FRVIAP152005', 'E000003'],
  ['FRELCP13050707', 'E000003'],
  ['FRMILP11437562', 'E000003']]

In [4]:
def surfaces_echangeurs(
    points_stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the convex hull surfaces of stations and echangeurs for each pool."""

    points = pd.concat([points_stations[[ID_POOL, "geometry"]], echangeurs[[ID_POOL, "geometry"]]], ignore_index=True)

    surfaces = points.to_crs(epsg=2154).dissolve(by=ID_POOL)
    surfaces["geometry"] = surfaces.geometry.convex_hull.buffer(BUFFER_ECHANGEUR)
  
    return surfaces.to_crs(epsg=4326).reset_index()

In [5]:
def init_pools(file_path: Path, filter: NaturePool = NaturePool.AIRE) -> gpd.GeoDataFrame:
    """Initialize pools from a geojson file."""
    #pools = pd.read_csv(pools_file).rename(columns={"id_aire": "id_pool"})
    pools = gpd.read_file(file_path).rename(columns={"id_aire": "id_pool"})
    match filter:
        case NaturePool.AIRE:
            return pools[pools[ID_POOL].str[:1] == "A"]
        case NaturePool.ECHANGEUR:
            return pools[pools[ID_POOL].str[:1] == "E"]
        case _:
            return pools

In [20]:
dmr_file = "../saturation/data_DMR/aires_dmr_2026-08-03.geojson"
echangeurs = init_pools(dmr_file, NaturePool.ECHANGEUR)
echangeurs

,id_pool,Aire,NomUsuel,Nature,GVO,Surface,Réseau,Axe,GVO_Axe,longitude,latitude,geometry
1871,E000001,NC,EchA0001Pr19,D,SANEF,NaN,C,A0001,None,2.530516,49.013634,POINT (2.53052 49.01363)
1872,E000002,NC,EchA0001Pr21,B,SANEF,NaN,C,A0001,None,2.540477,49.033849,POINT (2.54048 49.03385)
1873,E000003,NC,EchA0001Pr27,D,SANEF,NaN,C,A0001,None,2.551109,49.086427,POINT (2.55111 49.08643)
1874,E000004,NC,EchA0001Pr33,D,SANEF,NaN,C,A0001,None,2.562301,49.133518,POINT (2.5623 49.13352)
1875,E000005,NC,EchA0001Pr42,D,SANEF,NaN,C,A0001,None,2.611141,49.205605,POINT (2.61114 49.2056)
...,...,...,...,...,...,...,...,...,...,...,...,...
2952,E001082,NC,ECHA-RN205-E25-012712-0001,D,ATMB,NaN,C,N0205,None,6.765071,45.915615,POINT (6.76507 45.91562)
2953,E001083,NC,ECHA-RN205-E24-013909-0001,D,ATMB,NaN,C,N0205,None,6.761633,45.925860,POINT (6.76163 45.92586)
2954,E001084,NC,ECHA-RN205-E23-015057-0001,D,ATMB,NaN,C,N0205,None,6.748335,45.927374,POINT (6.74834 45.92737)
2955,E001085,NC,EchangeurN0523Pr0,D,ASF,NaN,C,N0523,None,1.794201,42.579609,POINT (1.7942 42.57961)


In [7]:
min_power = 50
statics = read_statics(day, min_power)
statics = statics[statics["puissance_nominale"] >= min_power]


In [ ]:
ech_station = stations_echangeurs(statics, echangeurs)

In [35]:
example_json = ech_station[ech_station[ID_POOL].isin(["E000001", "E000003"])].sort_values(by=ID_POOL).reset_index(drop=True)[[ID_STATION, ID_POOL]].to_json(orient="split", index=None)
example = json.loads(example_json)
# example

In [36]:
example_json = ech_station[ech_station[ID_POOL].isin(["E000001", "E000003"])].sort_values(by=ID_POOL).reset_index(drop=True)[[ID_STATION, ID_POOL]].to_json(orient="split", index=None)
example = json.loads(example_json)
#example

In [9]:
surf = surfaces_echangeurs(ech_station, echangeurs)
surf

,id_pool,geometry
0,E000001,"POLYGON ((2.51621 48.98923, 2.51609 48.98918, ..."
1,E000002,"POLYGON ((2.54184 49.03385, 2.54184 49.03377, ..."
2,E000003,"POLYGON ((2.55434 49.07163, 2.55432 49.07155, ..."
3,E000004,"POLYGON ((2.56367 49.13352, 2.56367 49.13344, ..."
4,E000005,"POLYGON ((2.6122 49.20618, 2.61228 49.2061, 2...."
...,...,...
1081,E001082,"POLYGON ((6.76636 45.91557, 6.76635 45.91548, ..."
1082,E001083,"POLYGON ((6.76292 45.92582, 6.76291 45.92573, ..."
1083,E001084,"POLYGON ((6.74962 45.92733, 6.74961 45.92724, ..."
1084,E001085,"POLYGON ((1.79542 42.57962, 1.79541 42.57953, ..."


In [16]:
refnat = {'location': [46.3, 2.3], 'zoom_start': 7, 'tiles': TILES_IGN, 'attr': ATTR_IGN}
#refnat  = {'tiles':'CartoDB positron', 'location': [46.3, 2.3], 'zoom_start': 7}
carte = surf.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)
##carte.save("aires.html")
#webbrowser.open(Path("aires.html").resolve().as_uri())
carte